# 5. LangGraph (Advanced) — 제품 탑재형 AI: Router → (Manual / Control / Troubleshoot / Handoff)
**LangGraph로 제품 탑재형 파이프라인(멀티 워크플로우)을 구성해보기**

---

## 🧩 시나리오(가상)
스마트홈/로보틱스/생활가전 탑재형 AI는 "무엇을 해야 하는지"를 먼저 분기해야 합니다.

- 매뉴얼 근거 기반 답변이 필요한가? (manual_info)
- 기기 제어를 실행해야 하는가? (device_control)
- 고장 진단 흐름이 필요한가? (troubleshooting)
- 안전/권한 이슈로 사람 연결이 필요한가? (handoff)

이 노트북은 LangGraph로 위 라우팅 + 각 워크플로우를 **하나의 그래프**로 묶고,
Streaming으로 내부 실행 흐름을 관찰합니다.

> 데이터/도구는 전부 더미이며, 실제 제품/현업 시스템과 무관합니다.


In [1]:
!pip -q install -U langgraph langchain langchain-openai pydantic

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/conda/lib/python3.10/site-packages/typing_extensions-4.15.0.dist-info/licenses/LICENSE'
Consider using the `--user` option or check the permissions.



In [2]:
import os, getpass, re, json
from typing import Annotated, Literal, Optional, List, Dict, Any
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

PROXY_URL = "http://10.0.4.135:8000/v1" 
PROXY_TOKEN = "1b2d4df88f5d71a9c55617796faf99f1" 


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, base_url=PROXY_URL, api_key=PROXY_TOKEN)


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) State 정의

messages 외에 intent를 상태로 관리합니다.


In [3]:
class GraphState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    intent: Optional[Literal["manual_info","device_control","troubleshooting","handoff"]]


## 2) Router 노드: 입력 의도 분류(구조화)

LLM에게 intent를 **4값(manual_info/device_control/troubleshooting/handoff)** 중 하나로만 출력하게 하여,
제품 탑재형 환경에서 중요한 **라우팅 정확도**를 올립니다.

- manual_info: 근거 기반(매뉴얼) 답변
- device_control: 기기 제어(설정 변경/시작/중지)
- troubleshooting: 에러코드/증상 진단
- handoff: 안전/권한 이슈로 사람 연결


In [4]:
class RouteDecision(BaseModel):
    intent: Literal["manual_info","device_control","troubleshooting","handoff"] = Field(description="사용자 의도 분류(제품 탑재형)")

router_llm = llm.with_structured_output(RouteDecision)

def router_node(state: GraphState):
    user_text = state["messages"][-1].content
    prompt = """다음 사용자 입력을 제품 탑재형 관점에서 4가지 의도 중 하나로 분류하세요.
- manual_info: 매뉴얼/기능 설명/주의사항 질문
- device_control: 기기 설정 변경/동작 시작/중지 등 제어 요청
- troubleshooting: 에러코드/증상 진단, 해결 절차 안내가 필요한 경우
- handoff: 안전 이슈(연기/과열/가스 냄새), 권한/개인정보 처리 등으로 사람 연결이 필요한 경우

입력:
""" + user_text
    decision = router_llm.invoke(prompt)
    return {"intent": decision.intent}


## 3) Tools 정의

### (A) Manual tool
- `search_manual(device_type, query)` : 더미 매뉴얼 스니펫 검색 + 출처 태그 반환

### (B) Device control tools
- `get_device_status(device)` : 상태 조회(더미)
- `set_device_setting(device, setting, value)` : 설정 변경(더미)

### (C) Troubleshooting tools
- `run_remote_diagnosis(device)` : 원격 진단(더미)
- `create_service_ticket(device, summary)` : 티켓 생성(더미)

> 실제 제품에서는 SDK/API/보안 게이트웨이를 통해 동작하며, 이 노트북은 구조만 체험합니다.


In [5]:
# --- 더미 매뉴얼/디바이스 데이터 ---
MANUAL_SNIPPETS = {
    "robot_vacuum": {
        "dock": "도킹이 안 되면: 도킹 앞 1m 장애물 제거, 충전단자 먼지 제거, 맵 재설정 고려(초기화 아님).",
        "bump": "충돌이 많으면: 거울/유광 바닥/조명에서 센서 오인식 가능. 청소 구역을 조정해보세요.",
    },
    "washing_machine": {
        "5c": "5C(배수) 에러: 배수 필터 이물질 제거, 배수 호스 꺾임/막힘 확인, 응급 배수 사용. 반복 시 점검 필요.",
    },
    "air_conditioner": {
        "filter": "필터 경고: 필터 세척/건조 후 장착, 필터 리셋 실행(모델별 메뉴 상이).",
        "temp": "권장 온도: 냉방 24~26도(환경에 따라 다름).",
    },
    "refrigerator": {
        "door": "문 열림 경고: 도어 패킹 이물질 제거, 문이 완전히 닫혔는지 확인.",
        "noise": "이상 소음: 수평 불량 시 진동음 증가. 수평 조절 다리 확인.",
    },
}

DEVICES = {
    "vacuum": {"device_type": "robot_vacuum", "status": "docked", "battery": 0.78, "mode": "idle"},
    "ac": {"device_type": "air_conditioner", "status": "online", "mode": "cool", "target_temp_c": 24},
    "washer": {"device_type": "washing_machine", "status": "online", "mode": "standby", "last_error": "5C"},
    "fridge": {"device_type": "refrigerator", "status": "online", "mode": "normal", "door_open": False},
}
TICKETS: List[dict] = []

# --- Manual tool ---
@tool
def search_manual(device_type: Literal["robot_vacuum","washing_machine","air_conditioner","refrigerator"], query: str) -> str:
    """제품 매뉴얼/가이드에서 query와 관련된 스니펫을 반환합니다(더미).
    - 근거 기반 답변이 필요할 때 사용하세요.
    - 반환 문자열에는 [manual:<device_type>:<key>] 출처 태그를 포함합니다.
    """
    q = query.lower()
    entries = MANUAL_SNIPPETS.get(device_type, {})
    hits = []
    for k,v in entries.items():
        if k in q or any(tok in q for tok in ["도킹","충돌","5c","배수","필터","온도","문","소음"]):
            hits.append(f"[manual:{device_type}:{k}] {v}")
    if not hits:
        return f"NO_HIT: {device_type}"
    return "\n".join(hits[:3])

# --- Device tools ---
@tool
def get_device_status(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """기기 상태(더미)를 조회합니다."""
    return json.dumps({"device": device, **DEVICES.get(device, {})}, ensure_ascii=False)

@tool
def set_device_setting(device: Literal["vacuum","ac","washer","fridge"], setting: str, value: Any) -> str:
    """기기 설정을 변경합니다(더미)."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"}, ensure_ascii=False)
    DEVICES[device][setting] = value
    return json.dumps({"ok": True, "device": device, "updated": {setting: value}}, ensure_ascii=False)

@tool
def run_remote_diagnosis(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """원격 자가진단(더미)"""
    if device == "washer" and DEVICES[device].get("last_error") == "5C":
        return json.dumps({"ok": True, "device": device, "diagnosis": "drain_issue_likely"}, ensure_ascii=False)
    return json.dumps({"ok": True, "device": device, "diagnosis": "no_critical_issue"}, ensure_ascii=False)

@tool
def create_service_ticket(device: Literal["vacuum","ac","washer","fridge"], summary: str) -> str:
    """서비스 티켓 생성(더미). 개인정보는 포함하지 않습니다."""
    t = {"ticket_id": f"TCK-{1000+len(TICKETS)}", "device": device, "summary": summary[:200]}
    TICKETS.append(t)
    return json.dumps({"ok": True, "ticket": t}, ensure_ascii=False)


## 4) 워크플로우별 Agent 노드

각 intent마다:
- LLM에 tools를 바인딩한 agent 노드
- ToolNode
- tools_condition으로 루프

### 주의
- 한 그래프에서 ToolNode를 3개 쓰기 때문에, 노드/툴 리스트를 분리해서 구성합니다.


In [6]:
# intent별 tool 세트
manual_tools = [search_manual]
control_tools = [get_device_status, set_device_setting]
troubleshoot_tools = [get_device_status, run_remote_diagnosis, search_manual, create_service_ticket]

manual_llm = llm.bind_tools(manual_tools)
control_llm = llm.bind_tools(control_tools)
troubleshoot_llm = llm.bind_tools(troubleshoot_tools)

def manual_agent_node(state: GraphState):
    sys = (
        "매뉴얼 질문에는 반드시 search_manual 도구를 호출하세요. "
        "최종 답변에는 [manual:<device_type>:<key>] 출처 태그를 문장/항목마다 포함하세요. "
        "근거가 없으면 '매뉴얼에 없음'이라고 답하세요."
    )
    resp = manual_llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def control_agent_node(state: GraphState):
    sys = (
        "기기 제어 요청에는 get_device_status로 현재 상태를 확인한 뒤, "
        "set_device_setting으로 변경하세요. "
        "불확실하면 실행하지 말고 짧게 확인 질문을 하세요."
    )
    resp = control_llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def troubleshoot_agent_node(state: GraphState):
    sys = (
        "고장/에러코드 문의에는 get_device_status + run_remote_diagnosis로 근거를 모으고, "
        "필요하면 search_manual로 해결 절차를 찾아 근거 태그와 함께 안내하세요. "
        "반복/위험/해결불가로 판단되면 create_service_ticket을 제안/생성하세요."
    )
    resp = troubleshoot_llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def handoff_node(state: GraphState):
    # handoff는 도구 없이 바로 종료
    msg = (
        "안전/권한/개인정보 이슈가 있어 바로 조치하기 어렵습니다. "
        "안전을 위해 전원을 차단하고(가능하면), 사람 상담/기사 연결로 안내드릴게요."
    )
    return {"messages": [HumanMessage(content=msg)]}

manual_tool_node = ToolNode(manual_tools)
control_tool_node = ToolNode(control_tools)
troubleshoot_tool_node = ToolNode(troubleshoot_tools)


## 5) Graph 구성 (Router → 서브플로우)

- START → router
- router 결과(intent)에 따라 각각의 서브플로우로 진입
- 각 서브플로우는 tools_condition으로 반복
- 최종은 END

또한 checkpointer를 붙여 thread_id 기반 메모리를 켭니다.


In [7]:
builder = StateGraph(GraphState)

builder.add_node("router", router_node)

builder.add_node("manual_agent", manual_agent_node)
builder.add_node("manual_tools", manual_tool_node)

builder.add_node("control_agent", control_agent_node)
builder.add_node("control_tools", control_tool_node)

builder.add_node("troubleshoot_agent", troubleshoot_agent_node)
builder.add_node("troubleshoot_tools", troubleshoot_tool_node)

builder.add_node("handoff", handoff_node)

# START → router
builder.add_edge(START, "router")

def route_from_intent(state: GraphState):
    return state.get("intent") or "manual_info"  # fallback

builder.add_conditional_edges(
    "router",
    route_from_intent,
    {
        "manual_info": "manual_agent",
        "device_control": "control_agent",
        "troubleshooting": "troubleshoot_agent",
        "handoff": "handoff",
    }
)

# manual flow
builder.add_conditional_edges("manual_agent", tools_condition, {"tools": "manual_tools", "__end__": END})
builder.add_edge("manual_tools", "manual_agent")

# control flow
builder.add_conditional_edges("control_agent", tools_condition, {"tools": "control_tools", "__end__": END})
builder.add_edge("control_tools", "control_agent")

# troubleshoot flow
builder.add_conditional_edges("troubleshoot_agent", tools_condition, {"tools": "troubleshoot_tools", "__end__": END})
builder.add_edge("troubleshoot_tools", "troubleshoot_agent")

# handoff flow (바로 종료)
builder.add_edge("handoff", END)

graph = builder.compile(checkpointer=InMemorySaver())


## 6) Streaming으로 실행 과정 관찰

노드별 업데이트를 보면서 Tool 호출이 어떤 흐름으로 일어나는지 확인합니다.


In [8]:
def stream_graph(user_input: str, thread_id: str = "t1"):
    config = {"configurable": {"thread_id": thread_id}}
    for event in graph.stream({"messages": [HumanMessage(content=user_input)], "intent": None}, config=config, stream_mode="updates"):
        print("\n--- event ---")
        for node, payload in event.items():
            print("node:", node)
            if isinstance(payload, dict) and "messages" in payload and payload["messages"]:
                msg = payload["messages"][-1]
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print(" tool_calls:", [tc["name"] for tc in msg.tool_calls])
                else:
                    print(" text:", str(msg.content)[:120])

# 예시
stream_graph("세탁기 5C 에러가 떠요. 어떻게 해야 하죠?") 



--- event ---
node: router

--- event ---
node: troubleshoot_agent
 tool_calls: ['get_device_status', 'run_remote_diagnosis']

--- event ---
node: troubleshoot_tools
 text: {"ok": true, "device": "washer", "diagnosis": "drain_issue_likely"}

--- event ---
node: troubleshoot_agent
 tool_calls: ['search_manual']

--- event ---
node: troubleshoot_tools
 text: [manual:washing_machine:5c] 5C(배수) 에러: 배수 필터 이물질 제거, 배수 호스 꺾임/막힘 확인, 응급 배수 사용. 반복 시 점검 필요.

--- event ---
node: troubleshoot_agent
 text: 세탁기에서 5C 에러가 발생했습니다. 이 에러는 배수 문제와 관련이 있습니다. 다음과 같은 조치를 취해보세요:

1. **배수 필터 이물질 제거**: 배수 필터에 이물질이 쌓여 있지 않은지 확인하고, 필요시 청소하세


## 📝 실습 과제

### 문제 1) device_control 플로우에 '위험 작업 확인' 추가 (상)
- 예: 공장초기화/펌웨어 업데이트 같은 위험 작업은 바로 실행하지 않고,
  - 사용자에게 확인 질문
  - confirmed=true가 들어온 다음 턴에서만 tool 호출
- State에 `confirmed: bool`을 추가하고, confirm 노드를 추가해보세요.

### 문제 2) manual_info 플로우에 'NO_HIT' 처리 강화 (중~상)
- `search_manual`이 NO_HIT를 반환하면,
  - agent가 "매뉴얼에 없음"이라고 답하고,
  - 상황/모델/증상 등 추가 질문 1~2개를 하도록 프롬프트/분기 로직을 보완해보세요.

### 문제 3) Subgraph 합성 (상)
- manual 흐름을 별도의 subgraph로 분리해,
  - 부모 그래프(router)에서 subgraph 노드를 호출하도록 바꿔보세요.

> 아래 셀은 학생용 TODO입니다.


In [9]:
# ✅ TODO 셀 (학생용)
# 1) Router를 업그레이드:
#    - 안전 키워드(연기/과열/가스/불꽃 등) 포함 시 handoff로 강제
# 2) device_control에서 '확인 단계'를 추가:
#    - 예: 'factory_reset' 같은 위험 작업은 별도 confirm 노드를 거쳐야만 tool 호출 가능하도록 그래프 확장
# 3) troubleshooting에서 티켓 생성 조건을 강화:
#    - 진단 결과가 drain_issue_likely 이고 사용자가 이미 필터/호스 확인했다고 하면 create_service_ticket 호출

# raise NotImplementedError("학생용 TODO 셀입니다. 구현 후 stream_graph로 동작 확인해보세요.")


In [10]:
# 예시 답안

from langchain_core.messages import AIMessage

# ----------------------------
# 공통 설정
# ----------------------------
SAFETY_KEYWORDS = ["연기", "과열", "가스", "불꽃", "화재", "타는 냄새", "누전", "스파크", "폭발"]
DANGEROUS_TOOLS = {"factory_reset", "update_firmware"}

def _is_yes(text: str) -> bool:
    t = text.strip().lower()
    yes = ["예", "네", "확인", "진행", "동의", "yes", "y", "ok", "okay"]
    return any(w in t for w in yes)

def _is_no(text: str) -> bool:
    t = text.strip().lower()
    no = ["아니", "취소", "중단", "no", "n"]
    return any(w in t for w in no)


# ----------------------------
# (문제 1) 위험 작업 tool 추가
# ----------------------------
@tool
def factory_reset(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """공장초기화(위험). 반드시 사용자 확인 후 실행하세요."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"}, ensure_ascii=False)
    # 더미: 상태를 리셋한 것처럼 표시
    DEVICES[device]["status"] = "reset_done"
    DEVICES[device]["mode"] = "idle"
    return json.dumps({"ok": True, "device": device, "action": "factory_reset"}, ensure_ascii=False)

@tool
def update_firmware(device: Literal["vacuum","ac","washer","fridge"], version: str) -> str:
    """펌웨어 업데이트(위험). 반드시 사용자 확인 후 실행하세요."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"}, ensure_ascii=False)
    DEVICES[device]["fw"] = version
    return json.dumps({"ok": True, "device": device, "action": "update_firmware", "version": version}, ensure_ascii=False)


# ----------------------------
# State 확장 (confirmed/pending_action)
# ----------------------------
class GraphState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    intent: Optional[Literal["manual_info","device_control","troubleshooting","handoff"]]
    confirmed: Optional[bool]
    pending_action: Optional[Dict[str, Any]]  # {"id":..., "name":..., "args":...}


# ----------------------------
# (문제 1) Router 업그레이드: 안전 키워드면 handoff 강제
# ----------------------------
def router_node_v2(state: GraphState):
    user_text = str(state["messages"][-1].content)
    if any(k in user_text for k in SAFETY_KEYWORDS):
        return {"intent": "handoff"}

    prompt = """다음 사용자 입력을 제품 탑재형 관점에서 4가지 의도 중 하나로 분류하세요.
- manual_info: 매뉴얼/기능 설명/주의사항 질문
- device_control: 기기 설정 변경/동작 시작/중지 등 제어 요청
- troubleshooting: 에러코드/증상 진단, 해결 절차 안내가 필요한 경우
- handoff: 안전 이슈(연기/과열/가스 냄새), 권한/개인정보 처리 등으로 사람 연결이 필요한 경우

입력:
""" + user_text
    decision = router_llm.invoke(prompt)
    return {"intent": decision.intent}


# ----------------------------
# (문제 2 + 3) manual subgraph (NO_HIT 강화 + 분리)
# ----------------------------
manual_tools_v2 = [search_manual]
manual_plan_llm = llm.bind_tools(manual_tools_v2)
manual_tool_node_v2 = ToolNode(manual_tools_v2)

def manual_plan_node(state: GraphState):
    sys = (
        "매뉴얼 질문에는 반드시 search_manual 도구를 1회 호출하세요. "
        "query에는 사용자의 질문 핵심 키워드를 넣으세요."
    )
    resp = manual_plan_llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def _last_search_manual_tool_content(state: GraphState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == "search_manual":
            return str(m.content)
    return ""

def manual_post_route(state: GraphState):
    content = _last_search_manual_tool_content(state)
    return "no_hit" if "NO_HIT" in content else "hit"

def manual_finalize_node(state: GraphState):
    sys = (
        "도구 결과(매뉴얼 스니펫)를 근거로 사용자에게 간단히 답하세요. "
        "가능하면 근거 태그([manual:...])를 그대로 포함하세요. "
        "추가 도구 호출은 하지 마세요."
    )
    resp = llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def manual_no_hit_node(state: GraphState):
    sys = (
        "search_manual 결과가 NO_HIT 입니다. "
        "1) '매뉴얼에 해당 내용이 없음'을 짧게 말하고, "
        "2) 상황/모델/증상/에러코드 등 추가 질문 1~2개를 하세요. "
        "3) 안전상 위험한 상황(연기/과열 등)이 의심되면 handoff를 권고하세요."
    )
    resp = llm.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

manual_builder = StateGraph(GraphState)
manual_builder.add_node("plan", manual_plan_node)
manual_builder.add_node("tools", manual_tool_node_v2)
manual_builder.add_node("finalize", manual_finalize_node)
manual_builder.add_node("no_hit", manual_no_hit_node)

manual_builder.add_edge(START, "plan")
manual_builder.add_conditional_edges("plan", tools_condition, {"tools": "tools", "__end__": END})
manual_builder.add_conditional_edges("tools", manual_post_route, {"hit": "finalize", "no_hit": "no_hit"})
manual_builder.add_edge("finalize", END)
manual_builder.add_edge("no_hit", END)

manual_subgraph = manual_builder.compile()


# ----------------------------
# device_control: confirm 단계 추가
# ----------------------------
control_tools_v2 = [get_device_status, set_device_setting, factory_reset, update_firmware]
control_llm_v2 = llm.bind_tools(control_tools_v2)
control_tool_node_v2 = ToolNode(control_tools_v2)

def _extract_first_dangerous_tool_call(state: GraphState) -> Optional[Dict[str, Any]]:
    last = state["messages"][-1]
    tcs = getattr(last, "tool_calls", []) or []
    for tc in tcs:
        if tc.get("name") in DANGEROUS_TOOLS:
            return {"id": tc.get("id", "pending"), "name": tc["name"], "args": tc.get("args", {})}
    return None

def confirm_request_node(state: GraphState):
    pending = _extract_first_dangerous_tool_call(state)
    if not pending:
        return {"messages": [AIMessage(content="(확인 단계) 위험 작업이 감지되지 않았습니다.")]}
    device = pending["args"].get("device", "unknown")
    action = pending["name"]
    msg = (
        f"⚠️ 요청하신 작업은 위험 작업(**{action}**)입니다.\n"
        f"- 대상 기기: {device}\n"
        "정말 진행할까요? 진행하려면 **'예'**, 취소하려면 **'아니오'** 라고 답해주세요."
    )
    return {"pending_action": pending, "confirmed": False, "messages": [AIMessage(content=msg)]}

def confirm_response_node(state: GraphState):
    pending = state.get("pending_action")
    user_text = str(state["messages"][-1].content)
    if not pending:
        return {"messages": [AIMessage(content="확인할 작업이 없습니다.")], "confirmed": False}

    if _is_yes(user_text):
        tc = {"id": pending.get("id", "pending"), "name": pending["name"], "args": pending.get("args", {})}
        return {"confirmed": True, "pending_action": None, "messages": [AIMessage(content="", tool_calls=[tc])]}

    if _is_no(user_text):
        return {"confirmed": False, "pending_action": None, "messages": [AIMessage(content="위험 작업을 취소했습니다.")]}

    return {"messages": [AIMessage(content="진행 여부가 불명확합니다. **'예'** 또는 **'아니오'**로 답해주세요.")]}

def control_agent_node_v2(state: GraphState):
    # ToolMessage 직후에는 요약만 하고 tool call을 하지 않도록 분기
    if isinstance(state["messages"][-1], ToolMessage):
        resp = llm.invoke([HumanMessage(content="도구 실행 결과를 사용자에게 2~3문장으로 요약하세요. 추가 도구 호출은 하지 마세요.")] + state["messages"])
        return {"messages": [resp], "confirmed": False, "pending_action": None}

    sys = (
        "기기 제어 요청에는 get_device_status로 현재 상태를 확인한 뒤 필요한 도구를 호출하세요.\n"
        "- 위험 작업(factory_reset, update_firmware)은 바로 실행되지 않고, confirm 단계에서만 실행됩니다.\n"
        "- 일반 설정 변경은 set_device_setting을 사용하세요."
    )
    resp = control_llm_v2.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def control_next(state: GraphState):
    last = state["messages"][-1]
    tcs = getattr(last, "tool_calls", None)
    if tcs:
        if _extract_first_dangerous_tool_call(state) and not (state.get("confirmed") or False):
            return "confirm_request"
        return "tools"
    return "__end__"


# ----------------------------
# troubleshooting: 티켓 생성 조건 강화
# ----------------------------
troubleshoot_tools_v2 = [get_device_status, run_remote_diagnosis, search_manual, create_service_ticket]
troubleshoot_llm_v2 = llm.bind_tools(troubleshoot_tools_v2)
troubleshoot_tool_node_v2 = ToolNode(troubleshoot_tools_v2)

def _latest_tool_json(state: GraphState, tool_name: str) -> Optional[dict]:
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == tool_name:
            try:
                return json.loads(m.content)
            except Exception:
                return None
    return None

def _user_checked_filter_hose(state: GraphState) -> bool:
    txt = ""
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            txt = str(m.content)
            break
    return ("필터" in txt or "배수" in txt) and ("호스" in txt) and ("확인" in txt or "점검" in txt or "봤" in txt)

def troubleshoot_agent_node_v2(state: GraphState):
    # ToolMessage 직후에는 요약만
    if isinstance(state["messages"][-1], ToolMessage):
        resp = llm.invoke([HumanMessage(content="도구 결과를 바탕으로 해결 절차를 체크리스트(3~5개)로 안내하세요.")] + state["messages"])
        return {"messages": [resp]}

    diag = _latest_tool_json(state, "run_remote_diagnosis")
    ticket = _latest_tool_json(state, "create_service_ticket")

    # 조건 충족 시 자동 티켓 생성
    if diag and diag.get("diagnosis") == "drain_issue_likely" and _user_checked_filter_hose(state) and not ticket:
        device = diag.get("device", "washer")
        summary = "배수 문제 가능성(drain_issue_likely). 사용자가 필터/호스 점검 완료. 추가 점검 필요."
        tc = {"id": "auto_ticket", "name": "create_service_ticket", "args": {"device": device, "summary": summary}}
        return {"messages": [AIMessage(content="", tool_calls=[tc])]}

    sys = (
        "고장/에러코드 문의에는 get_device_status + run_remote_diagnosis로 근거를 모으고, "
        "필요하면 search_manual로 해결 절차를 찾아 근거 태그와 함께 안내하세요. "
        "반복/위험/해결불가로 판단되면 create_service_ticket을 제안/생성하세요."
    )
    resp = troubleshoot_llm_v2.invoke([HumanMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}


def handoff_node_v2(state: GraphState):
    msg = (
        "🚨 안전/권한/개인정보 이슈가 있어 자동 조치가 어렵습니다.\n"
        "1) 가능하면 전원을 차단하고, 환기/안전 확보를 먼저 해주세요.\n"
        "2) 사람 상담/기사 연결로 안내드리겠습니다."
    )
    return {"messages": [AIMessage(content=msg)]}


# ----------------------------
# 부모 그래프 구성 (manual subgraph 합성 포함)
# ----------------------------
builder = StateGraph(GraphState)

builder.add_node("router", router_node_v2)
builder.add_node("manual_flow", manual_subgraph)

builder.add_node("control_agent", control_agent_node_v2)
builder.add_node("control_tools", control_tool_node_v2)
builder.add_node("confirm_request", confirm_request_node)
builder.add_node("confirm_response", confirm_response_node)

builder.add_node("troubleshoot_agent", troubleshoot_agent_node_v2)
builder.add_node("troubleshoot_tools", troubleshoot_tool_node_v2)

builder.add_node("handoff", handoff_node_v2)

def entry_route(state: GraphState):
    # pending_action이 있으면 confirm 응답 처리부터
    if state.get("pending_action"):
        return "await_confirm"
    return "router"

builder.add_conditional_edges(START, entry_route, {"await_confirm": "confirm_response", "router": "router"})

builder.add_conditional_edges(
    "router",
    lambda s: s.get("intent") or "manual_info",
    {
        "manual_info": "manual_flow",
        "device_control": "control_agent",
        "troubleshooting": "troubleshoot_agent",
        "handoff": "handoff",
    }
)
builder.add_edge("manual_flow", END)

# control flow
builder.add_conditional_edges("control_agent", control_next, {"confirm_request": "confirm_request", "tools": "control_tools", "__end__": END})
builder.add_edge("control_tools", "control_agent")
builder.add_edge("confirm_request", END)
builder.add_conditional_edges("confirm_response", tools_condition, {"tools": "control_tools", "__end__": END})

# troubleshoot flow
builder.add_conditional_edges("troubleshoot_agent", tools_condition, {"tools": "troubleshoot_tools", "__end__": END})
builder.add_edge("troubleshoot_tools", "troubleshoot_agent")

# handoff flow
builder.add_edge("handoff", END)

graph_v2 = builder.compile(checkpointer=InMemorySaver())


# ----------------------------
# 동작 확인: 멀티턴(확인 단계) 시나리오
# ----------------------------
def stream_graph_v2(user_input: str, thread_id: str = "t1"):
    config = {"configurable": {"thread_id": thread_id}}
    for event in graph_v2.stream({"messages": [HumanMessage(content=user_input)]}, config=config, stream_mode="updates"):
        print("\n--- event ---")
        for node, payload in event.items():
            print("node:", node)
            if isinstance(payload, dict) and payload.get("messages"):
                msg = payload["messages"][-1]
                if getattr(msg, "tool_calls", None):
                    print(" tool_calls:", [tc["name"] for tc in msg.tool_calls])
                else:
                    print(" text:", str(msg.content)[:160])



In [11]:
# 예시 실행
stream_graph_v2("세탁기 공장초기화 해줘", thread_id="demo_confirm")
stream_graph_v2("예", thread_id="demo_confirm")



--- event ---
node: router

--- event ---
node: control_agent
 tool_calls: ['get_device_status']

--- event ---
node: control_tools
 text: {"device": "washer", "device_type": "washing_machine", "status": "online", "mode": "standby", "last_error": "5C"}

--- event ---
node: control_agent
 text: 세탁기는 현재 온라인 상태이며 대기 모드에 있습니다. 마지막 오류는 "5C"로, 이는 배수 문제를 나타낼 수 있습니다. 공장 초기화를 원하시면 해당 오류를 해결한 후 진행하는 것이 좋습니다.

--- event ---
node: router

--- event ---
node: manual_flow
 text: 매뉴얼에 해당 내용이 없습니다. 

1) 세탁기의 공장 초기화 방법을 찾을 수 없었습니다. 
2) 현재 세탁기에서 어떤 증상이 발생하고 있나요? 또는 특정 에러 코드에 대해 더 알고 싶으신가요? 

또한, 만약 연기나 과열과 같은 안전상 위험한 상황이 의심된다면, 즉시 사용을 중지하고 
